In [ ]:
import numpy as np


class Params:
    def __init__(self, g=1.0, eps_d=1.0, kappa_b=6.0, kappa_a=0.3):
        self.g = g
        self.eps_d = eps_d
        self.kappa_b = kappa_b
        self.kappa_a = kappa_a
        self.c = np.sqrt(2.0) * g
        self.A0 = np.sqrt(eps_d / g)
        self.B0 = -np.sqrt(2.0) * eps_d / kappa_b
        self.delta = kappa_a * kappa_b / (4.0 * g**2)


def rhs(x, p: Params):
    """True vector field F(x), x=(A,B,PA,PB). Forward time = away from
    sources, toward sinks -- NOT the -F string-method convention."""
    A, B, PA, PB = x
    c, kb, ka = p.c, p.kappa_b, p.kappa_a
    Adot = c * (PA * PB - A * B) - (ka / 2) * A - ka * PA
    Bdot = (p.g / np.sqrt(2)) * (A**2 + PA**2) - p.eps_d / np.sqrt(2) - (kb / 2) * B - kb * PB
    PAdot = c * (B * PA - A * PB) + (ka / 2) * PA
    PBdot = c * A * PA + (kb / 2) * PB
    return np.array([Adot, Bdot, PAdot, PBdot])


def jacobian(x, p: Params):
    """Analytic Jacobian dF/dx, order (A,B,PA,PB). Verified against
    03_kappa_a_zero_structure.md's J_{W+} at kappa_a=0."""
    A, B, PA, PB = x
    c, kb, ka = p.c, p.kappa_b, p.kappa_a
    J = np.array([
        [-c * B - ka / 2,   -c * A,      c * PB - ka,   c * PA],
        [c * A,             -kb / 2,     c * PA,        -kb],
        [-c * PB,            c * PA,     c * B + ka / 2, -c * A],
        [c * PA,             0.0,        c * A,          kb / 2],
    ])
    return J


def H_energy(x, p: Params):
    """Conserved Hamiltonian, kappa_a != 0 case (should be 0 at all
    fixed points, and conserved along any trajectory)."""
    A, B, PA, PB = x
    g, eps_d, kb, ka = p.g, p.eps_d, p.kappa_b, p.kappa_a
    return (-np.sqrt(2) * g * A * B * PA
            + (g * A**2 / np.sqrt(2) - eps_d / np.sqrt(2) - kb * B / 2) * PB
            - kb / 2 * PB**2 + g / np.sqrt(2) * PA**2 * PB
            - ka / 2 * PA**2 - ka / 2 * A * PA)


def fixed_points(p: Params):
    """Exact closed-form fixed points for kappa_a != 0 (Wpm', S).
    M_+' is intentionally left out here -- not needed for the W+'->S leg,
    and its quadratic-root formula is in 01_model_and_notation.md if needed."""
    A0, delta, ka, g, c, kb = p.A0, p.delta, p.kappa_a, p.g, p.c, p.kappa_b
    assert delta < A0**2, "wells have merged (delta >= A0^2); Wpm' do not exist"
    Bshift = -ka / (2 * np.sqrt(2) * g)
    Wp = np.array([np.sqrt(A0**2 - delta), Bshift, 0.0, 0.0])
    Wm = np.array([-np.sqrt(A0**2 - delta), Bshift, 0.0, 0.0])
    S = np.array([0.0, p.B0, 0.0, 0.0])

    if ka!=0:
        a = (A0**2)/(2*A0**2 - 2*delta)*(1 + (2*delta)/(A0**2)-np.sqrt(1+4*(2*delta)/(A0**2)))
        P_A_value = np.sqrt(delta/(a*(a**2-1)))
        A_value = a*P_A_value
        B_value = -2*np.sqrt(2)*g/kb*A_value**2-ka/(2*np.sqrt(2)*g)
        P_B_value = -2*np.sqrt(2)*g*A_value*P_A_value/kb
        Mp = np.array([A_value, B_value, P_A_value, P_B_value])
    else : Mp = np.array([0.0, 0.0, A0, 0.0, 0.0])
    return Wp, Wm, S, Mp


In [ ]:
epsilon_p = 0.1
E_J = 37 * 2 * np.pi
phi_a = 0.11
phi_b = 0.204
alpha_sq = 8
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b
kappa_b = 5 / 10.4
kappa_2 = 4 * g**2 / kappa_b
kappa_a = 0.5 * kappa_2
eps_d = 2 * alpha_sq * g
target_name='S'
p = Params(g=g, eps_d=eps_d, kappa_b=kappa_b, kappa_a=kappa_a)
Wp, Wm, S, Mp = fixed_points(p)
target = {'S': S, 'Wm': Wm}[target_name]

In [ ]:
from scipy.integrate import solve_ivp
phi_1_array = np.linspace(0,np.pi, 10)
phi_2_array = np.linspace(0,np.pi, 10)
phi_3_array = np.linspace(3*np.pi/2, 2*np.pi, 20)

eps0 = 1e-2
T=120
R_escape = 10
fp = Wp


def ev(t, x):
    return R_escape - np.linalg.norm(x)
ev.terminal = True
ev.direction = -1
results = {}
for phi_1 in phi_1_array:
    results[phi_1] = {}
    for phi_2 in phi_2_array:
        results[phi_1][phi_2] = {}
        for phi_3 in phi_3_array:
            x0 = np.zeros(4)
            x0[0]= fp[0] + eps0 * np.cos(phi_1)
            x0[1]= fp[1] + eps0 * np.sin(phi_1)*np.cos(phi_2)
            x0[2]= fp[2] + eps0 * np.sin(phi_1)*np.sin(phi_2)*np.cos(phi_3)
            x0[3]= fp[3] + eps0 * np.sin(phi_1)*np.sin(phi_2)*np.sin(phi_3)

            try:
                sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
                                    dense_output=True, rtol=1e-8, atol=1e-10,
                                    events=ev)
                y = sol.y
                d2 = np.min(y[2]**2+y[3]**2)
            except Exception:
                continue
            results[phi_1][phi_2][phi_3] = (sol.y, d2)

In [ ]:
import matplotlib.pyplot as plt
for phi_1 in phi_1_array:
    for phi_2 in phi_2_array:
        for phi_3 in phi_3_array:
            if phi_3 not in results[phi_1][phi_2]:
                continue  # skip failed solves
            y,_ = results[phi_1][phi_2][phi_3]
            A, B, PA, PB = y[0], y[1], y[2], y[3]
            if np.max(PB)<.3:
                plt.plot(A, B)
plt.scatter([Wp[0]],[Wp[2]])
plt.scatter([S[0]],[S[2]])
plt.scatter([Mp[0]],[Mp[2]])
plt.xlim(-2,5)
plt.ylim(-2,5)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
for phi_1 in phi_1_array:
    for phi_2 in phi_2_array:
        for phi_3 in phi_3_array:
            if phi_3 not in results[phi_1][phi_2]:
                continue  # skip failed solves
            y,_ = results[phi_1][phi_2][phi_3]
            A, B, PA, PB = y[0], y[1], y[2], y[3]
            if np.max(PB)<.3:
                print(phi_1,phi_2,phi_3)
                plt.plot(A, PA)
plt.scatter([Wp[0]],[Wp[2]])
plt.scatter([S[0]],[S[2]])
plt.scatter([Mp[0]],[Mp[2]])
plt.xlim(-2,5)
plt.ylim(-2,5)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
for phi_1 in phi_1_array[5:]:
    for phi_2 in phi_2_array[5:6]:
        for phi_3 in phi_3_array[16:]:
            if phi_3 not in results[phi_1][phi_2]:
                continue  # skip failed solves
            y,_ = results[phi_1][phi_2][phi_3]
            A, B, PA, PB = y[0], y[1], y[2], y[3]
            if np.max(PB)<.3:
                print(phi_1,phi_2,phi_3)
                print(PA[-1])
                plt.plot(PA, PB)
plt.scatter([Wp[2]],[Wp[3]])
plt.scatter([S[2]],[S[3]])
plt.scatter([Mp[2]],[Mp[3]])
plt.scatter([-1.3],[-8])
#plt.xlim(-2,5)
#plt.ylim(-2,5)
plt.show()

In [ ]:
from scipy.integrate import solve_ivp
phi_1_array = np.linspace(1.79,1.81, 30)
phi_2_array = np.linspace(1.7,1.75, 20)
phi_3_array = np.linspace(5.9,6.0, 40)

eps0 = 1e-2
T=220
R_escape = 5
fp = Wp


def ev(t, x):
    return R_escape - np.linalg.norm(x)
ev.terminal = True
ev.direction = -1
results = {}
for phi_1 in phi_1_array:
    print(phi_1)
    results[phi_1] = {}
    for phi_2 in phi_2_array:
        results[phi_1][phi_2] = {}
        for phi_3 in phi_3_array:
            x0 = np.zeros(4)
            x0[0]= fp[0] + eps0 * np.cos(phi_1)
            x0[1]= fp[1] + eps0 * np.sin(phi_1)*np.cos(phi_2)
            x0[2]= fp[2] + eps0 * np.sin(phi_1)*np.sin(phi_2)*np.cos(phi_3)
            x0[3]= fp[3] + eps0 * np.sin(phi_1)*np.sin(phi_2)*np.sin(phi_3)

            try:
                sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
                                    dense_output=True, rtol=1e-8, atol=1e-10,
                                    events=ev)
                y = sol.y
                d2 = np.min(np.sum((y - target[:, None])**2, axis=0))
            except Exception:
                continue
            results[phi_1][phi_2][phi_3] = (sol.y, d2)

In [ ]:
import matplotlib.pyplot as plt
fig = plt.figure()
for phi_1 in phi_1_array:
    for phi_2 in phi_2_array:
        for phi_3 in phi_3_array:
            if phi_3 not in results[phi_1][phi_2]:
                continue  # skip failed solves
            y,d2 = results[phi_1][phi_2][phi_3]
            A, B, PA, PB = y[0], y[1], y[2], y[3]
            if d2<0.2:
                plt.plot(A, B)
plt.scatter([Wp[0]],[Wp[2]])
plt.scatter([S[0]],[S[2]])
plt.scatter([Mp[0]],[Mp[2]])
plt.xlim(-2,5)
plt.ylim(-3,5)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
fig = plt.figure()
for phi_1 in phi_1_array:
    for phi_2 in phi_2_array:
        for phi_3 in phi_3_array:
            if phi_3 not in results[phi_1][phi_2]:
                continue  # skip failed solves
            y,d2 = results[phi_1][phi_2][phi_3]
            A, B, PA, PB = y[0], y[1], y[2], y[3]
            if d2<0.1:
                plt.plot(A, PA)
plt.scatter([Wp[0]],[Wp[2]])
plt.scatter([S[0]],[S[2]])
plt.scatter([Mp[0]],[Mp[2]])
plt.xlim(-2,5)
plt.ylim(-2,5)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
fig = plt.figure()
for phi_1 in phi_1_array:
    for phi_2 in phi_2_array:
        for phi_3 in phi_3_array:
            if phi_3 not in results[phi_1][phi_2]:
                continue  # skip failed solves
            y,d2 = results[phi_1][phi_2][phi_3]
            A, B, PA, PB = y[0], y[1], y[2], y[3]
            if d2<0.1:
                plt.plot(PA, PB)
plt.scatter([Wp[2]],[Wp[3]])
plt.scatter([S[2]],[S[3]])
plt.scatter([Mp[2]],[Mp[3]])
#plt.xlim(-2,5)
#plt.ylim(-2,5)
plt.show()

In [ ]:
%matplotlib widget
fig = plt.figure()
ax = fig.add_subplot(projection='3d')

phi_1_plot, phi_2_plot, phi_3_plot, colors = [], [], [], []

for phi_1 in phi_1_array:
    for phi_2 in phi_2_array:
        for phi_3 in phi_3_array:
            if phi_3 not in results[phi_1][phi_2]:
                continue
            _,d2 = results[phi_1][phi_2][phi_3]
            if d2<0.1:
                phi_1_plot.append(phi_1)
                phi_2_plot.append(phi_2)
                phi_3_plot.append(phi_3)
                colors.append(d2)  # e.g. final value of coordinate 1

In [ ]:
# --- Plot ---
%matplotlib widget
fig = plt.figure()
ax = fig.add_subplot(projection='3d')

sc = ax.scatter(phi_1_plot, phi_2_plot, phi_3_plot, c=colors, cmap='RdBu_r', zorder=5)

# --- Fit plane via SVD ---
pts = np.array([phi_1_plot, phi_2_plot, phi_3_plot]).T  # shape (N, 3)
centroid = pts.mean(axis=0)
_, _, Vt = np.linalg.svd(pts - centroid)
normal = Vt[-1]  # normal to the plane (smallest singular value)
print(f"Plane normal: {normal},  equation: {normal[0]:.3f}(φ1-{centroid[0]:.3f}) + {normal[1]:.3f}(φ2-{centroid[1]:.3f}) + {normal[2]:.3f}(φ3-{centroid[2]:.3f}) = 0")
# Build a grid on the plane and plot it
p1, p2 = Vt[0], Vt[1]  # two vectors spanning the plane
x1, x2 = 1.79, 1.81
y1, y2 = 1.7,  1.75
z1, z2 = 5.9,  6.0

#ax.set_xlim(x1, x2)
#ax.set_ylim(y1, y2)
#ax.set_zlim(z1, z2)

# Build grid within the axis limits
t1 = np.linspace(x1 - centroid[0], x2 - centroid[0], 20)
t2 = np.linspace(y1 - centroid[1], y2 - centroid[1], 20)
u, v = np.meshgrid(t1, t2)
plane_pts = centroid + u[..., None] * p1 + v[..., None] * p2

# Clip to z limits
#z_plane = plane_pts[..., 2]
#mask = (z_plane >= z1) & (z_plane <= z2)
#plane_pts[~mask] = np.nan

ax.plot_surface(plane_pts[..., 0], plane_pts[..., 1], plane_pts[..., 2],
                alpha=0.3, color='gray')

ax.set_xlabel(r"$\phi_1$")
ax.set_ylabel(r"$\phi_2$")
ax.set_zlabel(r"$\phi_3$")
plt.colorbar(sc)
plt.tight_layout()
plt.show()

In [ ]:
def unstable_eigenbasis(fp, p: Params, tol=1e-8):
    """Return the two real vectors spanning the unstable eigenspace at
    fixed point `fp`. Handles both the real (node) case and the complex
    (focus) case -- see 05_numerics_and_open_questions.md's warning about
    naively taking .real of a conjugate pair."""
    J = jacobian(fp, p)
    w, v = np.linalg.eig(J)
    idx = np.where(w.real > tol)[0]
    if len(idx) != 2:
        raise RuntimeError(f"expected 2 unstable eigenvalues, got {len(idx)}: {w}")
    if np.abs(w[idx[0]].imag) < tol:
        # real case: two independent real eigenvectors
        u1 = v[:, idx[0]].real
        u2 = v[:, idx[1]].real
    else:
        # complex conjugate pair (focus/spiral case): use Re and Im of
        # ONE eigenvector as the two real basis vectors (they jointly
        # span the same real invariant plane)
        vec = v[:, idx[0]]
        u1 = vec.real
        u2 = vec.imag
    u1 = u1 / np.linalg.norm(u1)
    u2 = u2 / np.linalg.norm(u2)
    return u1, u2, w[idx]

u1, u2, w = unstable_eigenbasis(Wp, p)

In [ ]:
from scipy.integrate import solve_ivp
phi_1_array = np.linspace(1.5,2, 1000)#np.linspace(3.75312,3.75315, 1000)

eps0 = 1e-5
T=250
R_escape = 10
fp = Wp


def ev(t, x):
    return R_escape - np.linalg.norm(x)
ev.terminal = True
ev.direction = -1
results = {}
for phi_1 in phi_1_array:
    results[phi_1] = {}
    
    x0 = fp+eps0*np.cos(phi_1)*u1 + eps0*np.sin(phi_1)*u2
    
    try:
        sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
            dense_output=True, rtol=1e-8, atol=1e-10,
            events=ev)
        y = sol.y
        d2 = np.min(np.sum((y - target[:, None])**2, axis=0))
    except Exception:
        continue
    results[phi_1] = (sol.y, d2)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1,3, figsize=(20,10))
for phi_1 in phi_1_array:
    y,d2 = results[phi_1]
    A, B, PA, PB = y[0], y[1], y[2], y[3]
    axes[0].plot(A, PA)
    axes[1].plot(A, B)
    axes[2].plot(PA, PB)
axes[0].scatter([Wp[0]],[Wp[2]])
axes[0].scatter([S[0]],[S[2]])
axes[0].scatter([Mp[0]],[Mp[2]])
axes[1].scatter([Wp[0]],[Wp[1]])
axes[1].scatter([S[0]],[S[1]])
axes[1].scatter([Mp[0]],[Mp[1]])
axes[2].scatter([Wp[2]],[Wp[3]])
axes[2].scatter([S[2]],[S[3]])
axes[2].scatter([Mp[2]],[Mp[3]])
plt.show()


In [ ]:
d2_list = []
for phi_1 in phi_1_array:
    _,d2 = results[phi_1]
    d2_list.append(d2)
plt.scatter(phi_1_array, d2_list)
plt.xlabel("phi")
plt.ylabel("d_2")
plt.axhline(np.sum((Wp - S)**2))
plt.yscale("log")
plt.show()

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import minimize

p = Params(g=g, eps_d=eps_d, kappa_b=kappa_b, kappa_a=kappa_a)
Wp, Wm, S, Mp = fixed_points(p)
fp, target = Wp, S
u1, u2, w_unst = unstable_eigenbasis(fp, p)   # real basis of unstable plane at Wp

eps0, T, R_escape = 1e-5, 350, 10.0

def shoot(phi):
    x0 = fp + eps0 * (np.cos(phi) * u1 + np.sin(phi) * u2)
    sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
        dense_output=True, rtol=1e-12, atol=1e-12,
        events=ev)
    return np.min(np.sum((sol.y - target[:, None])**2, axis=0))

res = minimize(shoot, x0=[1.7], method='Nelder-Mead',
                options={'fatol': eps0})
phi_star = res.x[0]
print(phi_star, res.fun)

In [ ]:
x0 = fp + eps0 * (np.cos(phi_star) * u1 + np.sin(phi_star) * u2)
sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
    dense_output=True, rtol=1e-12, atol=1e-12,
    events=ev)
A,B,PA,PB=sol.y

fig, axes = plt.subplots(1,3, figsize=(20,10))
axes[0].plot(A, PA)
axes[1].plot(A, B)
axes[2].plot(PA, PB)
axes[0].scatter([Wp[0]],[Wp[2]])
axes[0].scatter([S[0]],[S[2]])
axes[0].scatter([Mp[0]],[Mp[2]])
axes[1].scatter([Wp[0]],[Wp[1]])
axes[1].scatter([S[0]],[S[1]])
axes[1].scatter([Mp[0]],[Mp[1]])
axes[2].scatter([Wp[2]],[Wp[3]])
axes[2].scatter([S[2]],[S[3]])
axes[2].scatter([Mp[2]],[Mp[3]])
axes[1].set_ylim(-4,1)
axes[2].set_ylim(-2,.5)
plt.show()


# Sweep for $\kappa_a$

In [ ]:
kappa_a_ratio_list = [1,.9, .7, .5, .3, .2, .15, .14, .13, .12, .11, .1, .09, .08, .07, .06, .05]
sol_list = []
phi_star_prev = 1.7
for kappa_a_ratio in kappa_a_ratio_list:
    p = Params(g=g, eps_d=eps_d, kappa_b=kappa_b, kappa_a=kappa_a*kappa_a_ratio)
    Wp, Wm, S, Mp = fixed_points(p)
    fp, target = Wp, S
    u1, u2, w_unst = unstable_eigenbasis(fp, p)   # real basis of unstable plane at Wp

    eps0, T, R_escape = 1e-5, 450, 10.0

    def shoot(phi):
        x0 = fp + eps0 * (np.cos(phi) * u1 + np.sin(phi) * u2)
        sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
            dense_output=True, rtol=1e-12, atol=1e-12,
            events=ev)
        return np.min(np.sum((sol.y - target[:, None])**2, axis=0))

    res = minimize(shoot, x0=[phi_star_prev], method='Nelder-Mead',
                    options={'fatol': eps0})
    phi_star = res.x[0]
    print(phi_star, res.fun)
    x0 = fp + eps0 * (np.cos(phi_star) * u1 + np.sin(phi_star) * u2)
    sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
        dense_output=True, rtol=1e-12, atol=1e-12,
        events=ev)
    sol_list.append(sol)
    phi_star_prev = phi_star

In [ ]:

A_0_p = np.sqrt(p.A0**2-p.delta)
fig, axes = plt.subplots(2,2, figsize=(15,15))
for i, sol in enumerate(sol_list[:-2]):
    y = sol.y  # shape (4, N)
    idx_end_string = np.argmin(np.sum((y - target[:, None])**2, axis=0))
    y_str = np.concatenate(
        [Wp[:, None], y[:, :idx_end_string], S[:, None]],
        axis=1
    )
    A, B, PA, PB = y_str
    axes[0,0].plot(A, PA, label=r"$\kappa_a/2\pi = $"+f"{kappa_a_ratio_list[i]*kappa_a/2/np.pi*1e3:.3e}"+" MHz")
    axes[0,1].plot(B, PB)
    axes[1,0].plot(A, B)
    axes[1,1].plot(PA, PB)
    
axes[0,0].scatter([Wp[0]],[Wp[2]])
axes[0,0].scatter([S[0]],[S[2]])
axes[0,0].scatter([Mp[0]],[Mp[2]])
axes[0,0].set_xlabel("A")
axes[0,0].set_ylabel("PA")
axes[0,0].legend()

axes[0,1].scatter([Wp[1]],[Wp[3]])
axes[0,1].scatter([S[1]],[S[3]])
axes[0,1].scatter([Mp[1]],[Mp[3]])
axes[0,1].set_xlabel("B")
axes[0,1].set_ylabel("PB")

axes[1,0].scatter([Wp[0]],[Wp[1]])
axes[1,0].scatter([S[0]],[S[1]])
axes[1,0].scatter([Mp[0]],[Mp[1]])
axes[1,0].set_xlabel("A")
axes[1,0].set_ylabel("B")

axes[1,1].scatter([Wp[2]],[Wp[3]])
axes[1,1].scatter([S[2]],[S[3]])
axes[1,1].scatter([Mp[2]],[Mp[3]])
axes[1,1].set_xlabel("PA")
axes[1,1].set_ylabel("PB")

plt.show()


In [ ]:
action_list = []
for (i,sol) in enumerate(sol_list[:-2]):
    y = sol.y  # shape (4, N)
    idx_end_string = np.argmin(np.sum((y - target[:, None])**2, axis=0))
    y_str = np.concatenate(
        [Wp[:, None], y[:, :idx_end_string], S[:, None]],
        axis=1
    )
    A, B, PA, PB = y_str
    dA = np.diff(A)
    dPA = np.diff(PA)
    dB = np.diff(B)
    dPB = np.diff(PB)
    ds = np.linalg.norm(np.diff(y_str, axis=0), axis=1)

    action = -np.sum(dA*PA[1:]-dPA*A[1:]+dB*PB[1:]-dPB*B[1:])
    action_list.append(action)
action_list = np.array(action_list)

In [ ]:
pert_th = np.array(kappa_a_ratio_list)*kappa_a/2/np.pi*1e3*p.A0**2*np.exp(-4*p.A0**2)+(np.array(kappa_a_ratio_list)*kappa_a/2/np.pi*1e3)**2/(2*kappa_2)*np.exp(-2*p.A0**2)

In [ ]:
kappa_ratio_keep = np.array(kappa_a_ratio_list[:-2])
delta_keep = kappa_ratio_keep*kappa_a*kappa_b/4/g**2
plt.scatter(kappa_ratio_keep*kappa_a/2/np.pi*1e3, -action_list)
plt.scatter([0],-p.A0**2)
plt.plot(kappa_ratio_keep*kappa_a/2/np.pi*1e3, -(p.A0**2*np.ones(len(kappa_ratio_keep))-delta_keep), ls='--', label=r"$-A_0'^2$")
plt.xlabel(r"$\kappa_a/2\pi$"+" (MHz)")
plt.ylabel(r"$S$")
plt.legend()
plt.show()

# Sweep for $\alpha^2$

In [ ]:
eps_d_ratio_list = [1,.9, .7, .5, .3, .25, .2, .17, .165, .164, .163, .162, .161, .16, .159, .158, .157, .156, .155, .154, .153, .152, .151, .15]
sol_list = []
phi_star_prev = 1.7
for eps_d_ratio in eps_d_ratio_list:
    p = Params(g=g, eps_d=eps_d*eps_d_ratio, kappa_b=kappa_b, kappa_a=kappa_a)
    Wp, Wm, S, Mp = fixed_points(p)
    fp, target = Wp, S
    u1, u2, w_unst = unstable_eigenbasis(fp, p)   # real basis of unstable plane at Wp

    eps0, T, R_escape = 1e-5, 1000, 10.0

    def shoot(phi):
        x0 = fp + eps0 * (np.cos(phi) * u1 + np.sin(phi) * u2)
        sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
            dense_output=True, rtol=1e-8, atol=1e-8,
            events=ev)
        return np.min(np.sum((sol.y - target[:, None])**2, axis=0))

    res = minimize(shoot, x0=[phi_star_prev], method='Nelder-Mead',
                    options={'fatol': eps0, 'maxiter' : 500})
    phi_star = res.x[0]
    print(phi_star, res.fun)
    x0 = fp + eps0 * (np.cos(phi_star) * u1 + np.sin(phi_star) * u2)
    sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
        dense_output=True, rtol=1e-8, atol=1e-8,
        events=ev)
    sol_list.append(sol)
    phi_star_prev = phi_star

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(15,15))
action_list = []
alpha_sq_list = []
for i, sol in enumerate(sol_list[:-1]):
    p = Params(g=g, eps_d=eps_d*eps_d_ratio_list[i], kappa_b=kappa_b, kappa_a=kappa_a)
    Wp, Wm, S, Mp = fixed_points(p)
    target=S
    y = sol.y  # shape (4, N)
    idx_end_string = np.argmin(np.sum((y - target[:, None])**2, axis=0))
    y_str = np.concatenate(
        [Wp[:, None], y[:, :idx_end_string], S[:, None]],
        axis=1
    )
    A, B, PA, PB = y_str
    dA = np.diff(A)
    dPA = np.diff(PA)
    dB = np.diff(B)
    dPB = np.diff(PB)
    ds = np.linalg.norm(np.diff(y_str, axis=0), axis=1)

    action = -np.sum(dA*PA[1:]-dPA*A[1:]+dB*PB[1:]-dPB*B[1:])
    action_list.append(action)
    alpha_sq_list.append(p.A0**2)
    axes[0,0].plot(A, PA, label=r"$\alpha^2 = $"+f"{p.A0**2:.3e}")
    axes[0,1].plot(B, PB)
    axes[1,0].plot(A, B)
    axes[1,1].plot(PA, PB)
    
    axes[0,0].scatter([Wp[0]],[Wp[2]])
    axes[0,0].scatter([S[0]],[S[2]])
    axes[0,0].scatter([Mp[0]],[Mp[2]])

    axes[0,1].scatter([Wp[1]],[Wp[3]])
    axes[0,1].scatter([S[1]],[S[3]])
    axes[0,1].scatter([Mp[1]],[Mp[3]])

    axes[1,0].scatter([Wp[0]],[Wp[1]])
    axes[1,0].scatter([S[0]],[S[1]])
    axes[1,0].scatter([Mp[0]],[Mp[1]])

    axes[1,1].scatter([Wp[2]],[Wp[3]])
    axes[1,1].scatter([S[2]],[S[3]])
    axes[1,1].scatter([Mp[2]],[Mp[3]])

axes[0,0].set_xlabel("A")
axes[0,0].set_ylabel("PA")
axes[0,0].legend()


axes[0,1].set_xlabel("B")
axes[0,1].set_ylabel("PB")


axes[1,0].set_xlabel("A")
axes[1,0].set_ylabel("B")


axes[1,1].set_xlabel("PA")
axes[1,1].set_ylabel("PB")
plt.show()


In [ ]:
from scipy.optimize import curve_fit

def func(x, a, b): return a*x + b

alpha_sq_list_keep=alpha_sq_list[-15:]
action_list_keep = action_list[-15:]
popt, pcov = curve_fit(func, np.array(alpha_sq_list_keep), -np.array(action_list_keep), p0=(-1, -1))

sc = plt.scatter(alpha_sq_list_keep, -np.array(action_list_keep),
            c=kappa_b <(4 * np.sqrt(2) * p.g * np.sqrt(np.array(alpha_sq_list_keep) - p.delta)))
plt.plot(alpha_sq_list_keep, func(np.array(alpha_sq_list_keep), *popt), label=f"{popt[0]:.3f}x + {popt[1]:.3f}")
plt.xlabel(r"$\alpha^2$")
plt.ylabel(r"$S$")
plt.colorbar(sc)
plt.legend()
plt.show()

In [ ]:
epsilon_p = 0.1
E_J = 37 * 2 * np.pi
phi_a = 0.11
phi_b = 0.204
alpha_sq = 8
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b
kappa_b = 5 / 10.4
kappa_2 = 4 * g**2 / kappa_b
kappa_a = 0.5 * kappa_2
eps_d = 2 * alpha_sq * g
target_name='S'
p = Params(g=g, eps_d=eps_d, kappa_b=kappa_b, kappa_a=kappa_a)
Wp, Wm, S, Mp = fixed_points(p)
target = {'S': S, 'Wm': Wm}[target_name]
u1, u2, w = unstable_eigenbasis(Wp, p)

In [ ]:
phi_1_array = np.linspace(0,2*np.pi, 200) #np.linspace(3.141602023045,3.14160202305, 1000)

eps0 = 1e-5
T=250
R_escape = 10
fp = Wp


def ev(t, x):
    return R_escape - np.linalg.norm(x)
ev.terminal = True
ev.direction = -1
results = {}
for phi_1 in phi_1_array:
    results[phi_1] = {}
    
    x0 = fp+eps0*np.cos(phi_1)*u1 + eps0*np.sin(phi_1)*u2
    
    try:
        sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
            dense_output=True, rtol=1e-8, atol=1e-10,
            events=ev)
        y = sol.y
        d2 = np.min(np.sum((y - target[:, None])**2, axis=0))
    except Exception:
        continue
    results[phi_1] = (sol.y, d2)

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(20,10))
for phi_1 in phi_1_array:
    y,d2 = results[phi_1]
    A, B, PA, PB = y[0], y[1], y[2], y[3]
    axes[0].plot(A, PA)
    axes[1].plot(A, B)
    axes[2].plot(PA, PB)
axes[0].scatter([Wp[0]],[Wp[2]])
axes[0].scatter([S[0]],[S[2]])
axes[0].scatter([Mp[0]],[Mp[2]])
axes[1].scatter([Wp[0]],[Wp[1]])
axes[1].scatter([S[0]],[S[1]])
axes[1].scatter([Mp[0]],[Mp[1]])
axes[2].scatter([Wp[2]],[Wp[3]])
axes[2].scatter([S[2]],[S[3]])
axes[2].scatter([Mp[2]],[Mp[3]])
#axes[0].set_xlim(.5,1.5)
#axes[0].set_ylim(-1,1)
#axes[2].set_xlim(-.25,1)
#axes[2].set_ylim(-.25,.25)
plt.show()


In [ ]:
fig, axes = plt.subplots(1,3, figsize=(20,10))
for phi_1 in phi_1_array:
    y,d2 = results[phi_1]
    A, B, PA, PB = y[0], y[1], y[2], y[3]
    axes[0].plot(A, PA)
    axes[1].plot(A, B)
    axes[2].plot(PA, PB)
axes[0].scatter([Wp[0]],[Wp[2]])
axes[0].scatter([S[0]],[S[2]])
axes[0].scatter([Mp[0]],[Mp[2]])
axes[1].scatter([Wp[0]],[Wp[1]])
axes[1].scatter([S[0]],[S[1]])
axes[1].scatter([Mp[0]],[Mp[1]])
axes[2].scatter([Wp[2]],[Wp[3]])
axes[2].scatter([S[2]],[S[3]])
axes[2].scatter([Mp[2]],[Mp[3]])
#axes[0].set_xlim(.5,1.5)
#axes[0].set_ylim(-1,1)
#axes[2].set_xlim(-.25,1)
#axes[2].set_ylim(-.25,.25)
plt.show()


In [ ]:
def h_Bmin(phi1, p, fp, u1, u2, eps0=1e-5, T=250, R_escape=10, B0=None):
    x0 = fp + eps0*np.cos(phi1)*u1 + eps0*np.sin(phi1)*u2

    def ev_escape(t, x):
        return R_escape - np.linalg.norm(x)
    ev_escape.terminal = True
    ev_escape.direction = -1

    sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
                     rtol=1e-11, atol=1e-13, events=ev_escape,
                     dense_output=False, max_step=0.05)

    Bmin = np.min(sol.y[1])
    return Bmin - B0

from scipy.optimize import brentq
f=lambda phi1: h_Bmin(phi1, p, Wp, u1, u2, B0=p.B0)
phi1_lo=0
phi1_hi=1.9*np.pi
phi1_star = brentq(f, phi1_lo, phi1_hi, xtol=1e-16, rtol=1e-15)
print(phi1_star)

In [ ]:
x0 = fp+eps0*np.cos(phi1_star)*u1 + eps0*np.sin(phi1_star)*u2
sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
    dense_output=True, rtol=1e-8, atol=1e-10,
    events=ev)
y = sol.y
print(np.min(sol.y[1])- p.B0)
fig, axes = plt.subplots(2,2, figsize=(15,15))
#p = Params(g=g, eps_d=eps_d*eps_d_ratio_list[i], kappa_b=kappa_b, kappa_a=kappa_a)
#Wp, Wm, S, Mp = fixed_points(p)
#target=S
#idx_end_string = np.argmin(np.sum((y - target[:, None])**2, axis=0))
#y_str = np.concatenate(
#    [Wp[:, None], y[:, :idx_end_string], S[:, None]],
#    axis=1
#)
A, B, PA, PB = y
axes[0,0].plot(A, PA)
axes[0,1].plot(B, PB)
axes[1,0].plot(A, B)
axes[1,1].plot(PA, PB)

x0 = fp+eps0*np.cos(3.141602023045)*u1 + eps0*np.sin(3.141602023045)*u2
sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
    dense_output=True, rtol=1e-8, atol=1e-10,
    events=ev)
y = sol.y
print(np.min(sol.y[1])- p.B0)
A, B, PA, PB = y
axes[0,0].plot(A, PA)
axes[0,1].plot(B, PB)
axes[1,0].plot(A, B)
axes[1,1].plot(PA, PB)

axes[0,0].scatter([Wp[0]],[Wp[2]])
axes[0,0].scatter([S[0]],[S[2]])
axes[0,0].scatter([Mp[0]],[Mp[2]])

axes[0,1].scatter([Wp[1]],[Wp[3]])
axes[0,1].scatter([S[1]],[S[3]])
axes[0,1].scatter([Mp[1]],[Mp[3]])

axes[1,0].scatter([Wp[0]],[Wp[1]])
axes[1,0].scatter([S[0]],[S[1]])
axes[1,0].scatter([Mp[0]],[Mp[1]])

axes[1,1].scatter([Wp[2]],[Wp[3]])
axes[1,1].scatter([S[2]],[S[3]])
axes[1,1].scatter([Mp[2]],[Mp[3]])

axes[0,0].set_xlabel("A")
axes[0,0].set_ylabel("PA")

axes[0,1].set_xlabel("B")
axes[0,1].set_ylabel("PB")


axes[1,0].set_xlabel("A")
axes[1,0].set_ylabel("B")


axes[1,1].set_xlabel("PA")
axes[1,1].set_ylabel("PB")
plt.show()


In [ ]:
eps_d_ratio_list = [.165]#, .165, .164, .163, .162, .161, .16, .159, .158, .157, .156, .155, .154, .153, .152, .151, .15]
sol_list = []
phi_star_prev = 1.7
for eps_d_ratio in eps_d_ratio_list:
    p = Params(g=g, eps_d=eps_d*eps_d_ratio, kappa_b=kappa_b, kappa_a=kappa_a)
    Wp, Wm, S, Mp = fixed_points(p)
    fp, target = Wp, S
    u1, u2, w_unst = unstable_eigenbasis(fp, p)   # real basis of unstable plane at Wp

    eps0, T, R_escape = 1e-5, 1000, 10.0

    def h_Bmin(phi1, p, fp, u1, u2, eps0=1e-5, T=250, R_escape=10, B0=None):
        x0 = fp + eps0*np.cos(phi1)*u1 + eps0*np.sin(phi1)*u2

        def ev_escape(t, x):
            return R_escape - np.linalg.norm(x)
        ev_escape.terminal = True
        ev_escape.direction = -1

        sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
                        rtol=1e-11, atol=1e-13, events=ev_escape,
                        dense_output=False, max_step=0.05)

        Bmin = np.min(sol.y[1])
        return Bmin - B0
    
    f=lambda phi1: h_Bmin(phi1, p, Wp, u1, u2, B0=p.B0)
    phi1_lo=0.9*np.pi
    phi1_hi=1.1*np.pi
    phi_star = brentq(f, phi1_lo, phi1_hi, xtol=1e-16, rtol=1e-15)
    print(phi_star)
    x0 = fp + eps0 * (np.cos(phi1_lo) * u1 + np.sin(phi1_lo) * u2)
    sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
        dense_output=True, rtol=1e-8, atol=1e-8,
        events=ev)
    sol_list.append(sol)
    x0 = fp + eps0 * (np.cos(phi_star) * u1 + np.sin(phi_star) * u2)
    sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
        dense_output=True, rtol=1e-8, atol=1e-8,
        events=ev)
    sol_list.append(sol)
    x0 = fp + eps0 * (np.cos(3.154567) * u1 + np.sin(3.154567) * u2)
    sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
        dense_output=True, rtol=1e-8, atol=1e-8,
        events=ev)
    sol_list.append(sol)
    x0 = fp + eps0 * (np.cos(phi1_hi) * u1 + np.sin(phi1_hi) * u2)
    sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
        dense_output=True, rtol=1e-8, atol=1e-8,
        events=ev)
    sol_list.append(sol)
    phi_star_prev = phi_star

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(15,15))
action_list = []
alpha_sq_list = []
for i, sol in enumerate(sol_list[:]):
    p = Params(g=g, eps_d=eps_d*eps_d_ratio_list[0], kappa_b=kappa_b, kappa_a=kappa_a)
    Wp, Wm, S, Mp = fixed_points(p)
    target=S
    y = sol.y  # shape (4, N)
    idx_end_string = np.argmin(np.sum((y - target[:, None])**2, axis=0))
    y_str = np.concatenate(
        [Wp[:, None], y[:, :idx_end_string], S[:, None]],
        axis=1
    )
    A, B, PA, PB = y#_str
    print(np.min(sol.y[1])-p.B0)
    dA = np.diff(A)
    dPA = np.diff(PA)
    dB = np.diff(B)
    dPB = np.diff(PB)
    ds = np.linalg.norm(np.diff(y_str, axis=0), axis=1)

    action = -np.sum(dA*PA[1:]-dPA*A[1:]+dB*PB[1:]-dPB*B[1:])
    action_list.append(action)
    alpha_sq_list.append(p.A0**2)
    axes[0,0].plot(A, PA, label=r"$\alpha^2 = $"+f"{p.A0**2:.3e}")
    axes[0,1].plot(B, PB)
    axes[1,0].plot(A, B)
    axes[1,1].plot(PA, PB)
    
    axes[0,0].scatter([Wp[0]],[Wp[2]])
    axes[0,0].scatter([S[0]],[S[2]])
    axes[0,0].scatter([Mp[0]],[Mp[2]])

    axes[0,1].scatter([Wp[1]],[Wp[3]])
    axes[0,1].scatter([S[1]],[S[3]])
    axes[0,1].scatter([Mp[1]],[Mp[3]])

    axes[1,0].scatter([Wp[0]],[Wp[1]])
    axes[1,0].scatter([S[0]],[S[1]])
    axes[1,0].scatter([Mp[0]],[Mp[1]])

    axes[1,1].scatter([Wp[2]],[Wp[3]])
    axes[1,1].scatter([S[2]],[S[3]])
    axes[1,1].scatter([Mp[2]],[Mp[3]])

axes[0,0].set_xlabel("A")
axes[0,0].set_ylabel("PA")
axes[0,0].legend()


axes[0,1].set_xlabel("B")
axes[0,1].set_ylabel("PB")


axes[1,0].set_xlabel("A")
axes[1,0].set_ylabel("B")


axes[1,1].set_xlabel("PA")
axes[1,1].set_ylabel("PB")
plt.show()


In [ ]:
p = Params(g=g, eps_d=eps_d, kappa_b=kappa_b, kappa_a=kappa_a)
Wp, Wm, S, Mp = fixed_points(p)
fp, target = Wp, S
u1, u2, w_unst = unstable_eigenbasis(fp, p)   # real basis of unstable plane at Wp

eps0, T, R_escape = 1e-5, 250, 10.0

def shoot(phi):
    x0 = fp + eps0 * (np.cos(phi) * u1 + np.sin(phi) * u2)
    sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
        dense_output=True, rtol=1e-8, atol=1e-8,
        events=ev)
    return np.min(np.sum((sol.y - target[:, None])**2, axis=0))

res = minimize(shoot, x0=[3.141602023045], method='Nelder-Mead',
                options={'fatol': 1e-10, 'maxiter':2000})
phi_star = res.x[0]
print(phi_star, res.fun)

# Sweep for $\kappa_b$

In [ ]:
kappa_b_ratio_list_bare = [1,1.2,1.4,1.6,1.8,2,2.2,2.4,2.6,2.8,3,1,.9,.7,.5,.3,.2]
kappa_b_ratio_list = []
sol_list = []
phi_star_prev = 1.7
for kappa_b_ratio in kappa_b_ratio_list_bare:
    if kappa_b_ratio == 1: phi_star_prev = 1.7
    p = Params(g=g, eps_d=eps_d, kappa_b=kappa_b*kappa_b_ratio, kappa_a=kappa_a)
    Wp, Wm, S, Mp = fixed_points(p)
    fp, target = Wp, S
    u1, u2, w_unst = unstable_eigenbasis(fp, p)   # real basis of unstable plane at Wp

    eps0, T, R_escape = 1e-5, 450, 10.0

    def shoot(phi):
        x0 = fp + eps0 * (np.cos(phi) * u1 + np.sin(phi) * u2)
        sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
            dense_output=True, rtol=1e-12, atol=1e-12,
            events=ev)
        return np.min(np.sum((sol.y - target[:, None])**2, axis=0))

    res = minimize(shoot, x0=[phi_star_prev], method='Nelder-Mead',
                    options={'fatol': eps0})
    phi_star = res.x[0]
    print(phi_star, res.fun)
    x0 = fp + eps0 * (np.cos(phi_star) * u1 + np.sin(phi_star) * u2)
    sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
        dense_output=True, rtol=1e-12, atol=1e-12,
        events=ev)
    if res.fun < .1: 
        sol_list.append(sol)
        kappa_b_ratio_list.append(kappa_b_ratio)
    phi_star_prev = phi_star

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(20,10))
action_list = []
for i, sol in enumerate(sol_list):
    p = Params(g=g, eps_d=eps_d, kappa_b=kappa_b*kappa_b_ratio_list[i], kappa_a=kappa_a)
    Wp, Wm, S, Mp = fixed_points(p)
    target=S
    y = sol.y  # shape (4, N)
    idx_end_string = np.argmin(np.sum((y - target[:, None])**2, axis=0))
    y_str = np.concatenate(
        [Wp[:, None], y[:, :idx_end_string], S[:, None]],
        axis=1
    )
    A, B, PA, PB = y_str
    dA = np.diff(A)
    dPA = np.diff(PA)
    dB = np.diff(B)
    dPB = np.diff(PB)
    ds = np.linalg.norm(np.diff(y_str, axis=0), axis=1)

    action = -np.sum(dA*PA[1:]-dPA*A[1:]+dB*PB[1:]-dPB*B[1:])
    action_list.append(action)
    axes[0].plot(A, PA, label=r"$\kappa_b/2\pi = $"+f"{kappa_b*kappa_b_ratio_list[i]/2/np.pi*1e3:.3e}"+" MHz")
    axes[1].plot(A, B)
    axes[2].plot(PA, PB)
    axes[0].scatter([Wp[0]],[Wp[2]])
    axes[0].scatter([S[0]],[S[2]])
    axes[0].scatter([Mp[0]],[Mp[2]])
    axes[1].scatter([Wp[0]],[Wp[1]])
    axes[1].scatter([S[0]],[S[1]])
    axes[1].scatter([Mp[0]],[Mp[1]])
    axes[2].scatter([Wp[2]],[Wp[3]])
    axes[2].scatter([S[2]],[S[3]])
    axes[2].scatter([Mp[2]],[Mp[3]])
axes[1].set_ylim(-4,4)
axes[2].set_ylim(-2,.5)
axes[0].legend()
plt.show()


In [ ]:
kappa_b_ratio_arr = np.array(kappa_b_ratio_list)

plt.scatter(
    kappa_b_ratio_arr * kappa_b / 2 / np.pi * 1e3,
    -np.array(action_list),
    c=kappa_b_ratio_arr*kappa_b < 4 * np.sqrt(2) * p.g * np.sqrt(p.A0**2 - p.kappa_a * kappa_b_ratio_arr*kappa_b / 4 / p.g**2)
)
plt.xlabel(r"$\kappa_b/2\pi$"+" (MHz)")
plt.ylabel(r"$S$")
plt.show()